# Road Damage Detection — 04. Final Comparison, Benchmark, and Export

Run this notebook only after the three candidates are frozen. It uses the untouched test split to
calculate precision, recall, F1, mAP, per-class AP, matched-box IoU, batch-1 latency/FPS, model size,
parameter count, peak VRAM, ONNX export, and the accuracy-efficiency Pareto frontier.

It prefers a tuned checkpoint and falls back to the baseline if tuning has not been completed.

## Test-set rule

Do not repeatedly change training decisions after looking at these results. Any new decision based
on the test set weakens its value as an unbiased final estimate.

In [ ]:
# Versions pinned for reproducibility and YOLO26 support.
%pip install -q \
    ultralytics==8.4.92 \
    roboflow==1.3.13 \
    pyyaml==6.0.3 \
    pandas>=2.2,<3 \
    matplotlib>=3.9,<4 \
    pillow>=11,<13 \
    psutil>=6,<8 \
    onnx>=1.17,<2 \
    onnxruntime>=1.20,<2

In [ ]:
from __future__ import annotations

import gc
import json
import os
import platform
import random
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from PIL import Image

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
else:
    raise RuntimeError(
        'No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU.'
    )

## Dataset preflight

This notebook uses only the processed object-detection dataset generated by notebook 00:

```text
/content/drive/MyDrive/RoadDamageYOLO/data/road-damage-detection-bbox-v1
```

It copies the dataset to Colab's local disk and creates a runtime-specific YAML that points to:

```text
/content/road-damage-detection-bbox-v1
```

Do not change these paths back to `road-damage-1`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/RoadDamageYOLO")

# This is the processed object-detection dataset produced by notebook 00.
PERSISTENT_DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "road-damage-detection-bbox-v1"
)

# Training from Colab's local disk is faster than reading thousands of files
# directly from Google Drive.
LOCAL_DATASET_DIR = Path(
    "/content/road-damage-detection-bbox-v1"
)

RUNS_ROOT = PROJECT_ROOT / "runs"
REPORTS_ROOT = PROJECT_ROOT / "reports"
EXPORTS_ROOT = PROJECT_ROOT / "exports"

for directory in [
    RUNS_ROOT,
    REPORTS_ROOT,
    EXPORTS_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Persistent dataset:", PERSISTENT_DATASET_DIR)
print("Local dataset:", LOCAL_DATASET_DIR)

In [ ]:
def find_validation_folder(dataset_dir: Path) -> str:
    if (dataset_dir / "valid").exists():
        return "valid"
    if (dataset_dir / "val").exists():
        return "val"
    raise FileNotFoundError(
        f"Neither 'valid' nor 'val' exists inside {dataset_dir}."
    )


def verify_dataset_structure(dataset_dir: Path) -> dict[str, int]:
    validation_folder = find_validation_folder(dataset_dir)

    split_folders = {
        "train": "train",
        "val": validation_folder,
        "test": "test",
    }

    counts = {}
    missing_paths = []

    for split_name, folder_name in split_folders.items():
        images_dir = dataset_dir / folder_name / "images"
        labels_dir = dataset_dir / folder_name / "labels"

        if not images_dir.exists():
            missing_paths.append(str(images_dir))
        if not labels_dir.exists():
            missing_paths.append(str(labels_dir))

        if images_dir.exists():
            counts[split_name] = sum(
                1
                for path in images_dir.iterdir()
                if path.is_file()
                and path.suffix.lower()
                in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            )

    if missing_paths:
        raise FileNotFoundError(
            "The processed dataset is incomplete. Missing:\n"
            + "\n".join(missing_paths)
        )

    if not all(counts.get(split, 0) > 0 for split in ["train", "val", "test"]):
        raise RuntimeError(
            f"One or more dataset splits contain no images: {counts}"
        )

    return counts


def copy_dataset_to_local(
    persistent_dir: Path = PERSISTENT_DATASET_DIR,
    local_dir: Path = LOCAL_DATASET_DIR,
    force_refresh: bool = False,
) -> None:
    if not persistent_dir.exists():
        raise FileNotFoundError(
            f"Processed dataset not found at:\n{persistent_dir}\n\n"
            "Run 00_Data_Setup_and_Audit_UPDATED.ipynb first."
        )

    persistent_counts = verify_dataset_structure(persistent_dir)
    print("Persistent dataset image counts:", persistent_counts)

    local_is_ready = False
    if local_dir.exists() and not force_refresh:
        try:
            local_counts = verify_dataset_structure(local_dir)
            local_is_ready = local_counts == persistent_counts
            if local_is_ready:
                print("Using the existing verified local dataset copy.")
        except Exception:
            local_is_ready = False

    if not local_is_ready:
        if local_dir.exists():
            print("Removing stale or incomplete local dataset copy...")
            shutil.rmtree(local_dir)

        print("Copying processed dataset from Google Drive to Colab local storage...")
        shutil.copytree(persistent_dir, local_dir)

        local_counts = verify_dataset_structure(local_dir)
        if local_counts != persistent_counts:
            raise RuntimeError(
                "Local copy verification failed.\n"
                f"Persistent counts: {persistent_counts}\n"
                f"Local counts: {local_counts}"
            )

    print("Local dataset ready:", local_dir)


def build_local_data_yaml(
    dataset_dir: Path = LOCAL_DATASET_DIR,
) -> Path:
    validation_folder = find_validation_folder(dataset_dir)

    yaml_candidates = [
        dataset_dir / "data_detection_drive.yaml",
        dataset_dir / "data_detection.yaml",
        dataset_dir / "data.yaml",
    ]

    source_yaml = next(
        (path for path in yaml_candidates if path.exists()),
        None,
    )

    if source_yaml is None:
        raise FileNotFoundError(
            f"No dataset YAML was found inside {dataset_dir}."
        )

    with source_yaml.open("r", encoding="utf-8") as file:
        source_config = yaml.safe_load(file)

    names = source_config.get("names")
    if not names:
        raise ValueError(
            f"The dataset YAML {source_yaml} does not contain class names."
        )

    local_config = {
        "path": str(dataset_dir.resolve()),
        "train": "train/images",
        "val": f"{validation_folder}/images",
        "test": "test/images",
        "names": names,
    }

    output_yaml = dataset_dir / "data_colab.yaml"
    with output_yaml.open("w", encoding="utf-8") as file:
        yaml.safe_dump(
            local_config,
            file,
            sort_keys=False,
            allow_unicode=True,
        )

    return output_yaml


# Set to True only when the local copy is stale or after changing dataset version.
FORCE_DATASET_REFRESH = False

copy_dataset_to_local(force_refresh=FORCE_DATASET_REFRESH)
DATA_YAML = build_local_data_yaml()

image_counts = verify_dataset_structure(LOCAL_DATASET_DIR)

print("\nVerified local dataset counts:")
for split, count in image_counts.items():
    print(f"  {split}: {count}")

print("\nTraining/evaluation YAML:")
print(DATA_YAML.read_text())

assert str(LOCAL_DATASET_DIR) in DATA_YAML.read_text(), (
    "The local dataset YAML does not point to the local processed dataset."
)

In [ ]:
environment = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'ultralytics': ultralytics.__version__,
}
(REPORTS_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)
environment

## 1. Locate frozen checkpoints

In [ ]:
from ultralytics import YOLO

MODEL_SPECS = {
    'YOLOv8s': {
        'final': RUNS_ROOT / 'YOLOv8s' / 'final_tuned_seed42' / 'weights' / 'best.pt',
        'baseline': RUNS_ROOT / 'YOLOv8s' / 'baseline_seed42' / 'weights' / 'best.pt',
    },
    'YOLO11s': {
        'final': RUNS_ROOT / 'YOLO11s' / 'final_tuned_seed42' / 'weights' / 'best.pt',
        'baseline': RUNS_ROOT / 'YOLO11s' / 'baseline_seed42' / 'weights' / 'best.pt',
    },
    'YOLO26s': {
        'final': RUNS_ROOT / 'YOLO26s' / 'final_tuned_seed42' / 'weights' / 'best.pt',
        'baseline': RUNS_ROOT / 'YOLO26s' / 'baseline_seed42' / 'weights' / 'best.pt',
    },
}

CHECKPOINTS = {}
for model_name, paths in MODEL_SPECS.items():
    if paths['final'].exists():
        CHECKPOINTS[model_name] = paths['final']
    elif paths['baseline'].exists():
        CHECKPOINTS[model_name] = paths['baseline']
    else:
        print('Missing checkpoint:', model_name)

if not CHECKPOINTS:
    raise FileNotFoundError('No trained checkpoints were found.')

for name, checkpoint in CHECKPOINTS.items():
    print(name, '→', checkpoint)

## 2. Frozen evaluation configuration

In [ ]:
DEVICE = 0
IMGSZ = 640
VAL_BATCH = 16
WORKERS = 2

# Select these using validation data before considering the test results final.
DEPLOY_CONF = 0.25
MATCH_IOU_THRESHOLD = 0.50

MAX_IOU_IMAGES = None      # None = full test split; use a small number only for a smoke test.
BENCHMARK_IMAGES = 150
WARMUP_RUNS = 10
RUN_ONNX_EXPORT = True
VALIDATE_ONNX = False

COMPARISON_DIR = REPORTS_ROOT / 'final_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

## 3. Pre-register model-selection inputs from validation data

The production model is selected from validation metrics plus the hardware benchmark. Test metrics
are reported later but are not used by the selection rule.

In [ ]:
validation_rows = []
for model_name, checkpoint in CHECKPOINTS.items():
    model_dir = RUNS_ROOT / model_name
    summary_name = (
        'final_tuned_validation_summary.json'
        if 'final_tuned' in str(checkpoint)
        else 'baseline_validation_summary.json'
    )
    summary_path = model_dir / summary_name

    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        validation_precision = float(summary['precision'])
        validation_recall = float(summary['recall'])
        validation_map50 = float(summary['map50'])
        validation_map50_95 = float(summary['map50_95'])
    else:
        print(f'Validation summary missing for {model_name}; evaluating validation split now...')
        validation_metrics = YOLO(str(checkpoint)).val(
            data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=VAL_BATCH,
            device=DEVICE, workers=WORKERS, plots=False,
        )
        validation_precision = float(validation_metrics.box.mp)
        validation_recall = float(validation_metrics.box.mr)
        validation_map50 = float(validation_metrics.box.map50)
        validation_map50_95 = float(validation_metrics.box.map)

    validation_rows.append({
        'model': model_name,
        'validation_precision': validation_precision,
        'validation_recall': validation_recall,
        'validation_map50': validation_map50,
        'validation_map50_95': validation_map50_95,
        'model_size_mb': checkpoint.stat().st_size / (1024 ** 2),
    })

validation_df = pd.DataFrame(validation_rows).sort_values(
    'validation_map50_95', ascending=False
)
display(validation_df)

## 4. Official Ultralytics test metrics

In [ ]:
def names_dict(names):
    return {int(key): str(value) for key, value in names.items()} if isinstance(names, dict) else dict(enumerate(names))

test_rows, per_class_rows = [], []
for model_name, checkpoint in CHECKPOINTS.items():
    print(f'\nEvaluating {model_name}...')
    model = YOLO(str(checkpoint))
    metrics = model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=VAL_BATCH,
        device=DEVICE, workers=WORKERS, plots=True,
        project=str(COMPARISON_DIR / 'test_runs'), name=model_name,
    )
    precision, recall = float(metrics.box.mp), float(metrics.box.mr)
    test_rows.append({
        'model': model_name,
        'checkpoint_stage': 'final_tuned' if 'final_tuned' in str(checkpoint) else 'baseline',
        'precision': precision, 'recall': recall,
        'f1': 2 * precision * recall / (precision + recall + 1e-12),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'parameters': int(sum(parameter.numel() for parameter in model.model.parameters())),
        'model_size_mb': checkpoint.stat().st_size / (1024 ** 2),
        'val_preprocess_ms': float(metrics.speed.get('preprocess', np.nan)),
        'val_inference_ms': float(metrics.speed.get('inference', np.nan)),
        'val_postprocess_ms': float(metrics.speed.get('postprocess', np.nan)),
        'checkpoint': str(checkpoint),
    })
    names = names_dict(model.names)
    maps = [float(value) for value in metrics.box.maps]
    for class_id, class_name in names.items():
        if class_id < len(maps):
            per_class_rows.append({
                'model': model_name, 'class_id': class_id,
                'class_name': class_name, 'map50_95': maps[class_id],
            })

test_metrics_df = pd.DataFrame(test_rows).sort_values('map50_95', ascending=False)
per_class_df = pd.DataFrame(per_class_rows)
display(test_metrics_df)
display(per_class_df.pivot(index='class_name', columns='model', values='map50_95'))

## 5. Matched-box IoU diagnostic

mAP already evaluates detections at multiple IoU thresholds. This additional diagnostic filters
predictions at the deployment confidence, greedily matches predictions and ground truth within the
same class, and reports IoU only for matched true positives at IoU ≥ 0.50.

In [ ]:
def box_iou_xyxy(a: np.ndarray, b: np.ndarray) -> float:
    x1, y1 = max(float(a[0]), float(b[0])), max(float(a[1]), float(b[1]))
    x2, y2 = min(float(a[2]), float(b[2])), min(float(a[3]), float(b[3]))
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def ground_truth(image_path: Path):
    label_path = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    with Image.open(image_path) as image:
        width, height = image.size
    boxes, classes = [], []
    if label_path.exists():
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:])
            boxes.append([
                (x - w / 2) * width, (y - h / 2) * height,
                (x + w / 2) * width, (y + h / 2) * height,
            ])
            classes.append(class_id)
    return np.asarray(boxes, dtype=float).reshape(-1, 4), np.asarray(classes, dtype=int)


def matched_ious(pred_boxes, pred_classes, pred_conf, gt_boxes, gt_classes, threshold):
    used_gt, ious_out, classes_out = set(), [], []
    for pred_index in np.argsort(-pred_conf):
        candidates = [
            index for index, gt_class in enumerate(gt_classes)
            if index not in used_gt and int(gt_class) == int(pred_classes[pred_index])
        ]
        if not candidates:
            continue
        values = [box_iou_xyxy(pred_boxes[pred_index], gt_boxes[index]) for index in candidates]
        position = int(np.argmax(values))
        gt_index, best_iou = candidates[position], float(values[position])
        if best_iou >= threshold:
            used_gt.add(gt_index)
            ious_out.append(best_iou)
            classes_out.append(int(pred_classes[pred_index]))
    return ious_out, classes_out

In [ ]:
test_images = sorted(
    path for path in (LOCAL_DATASET_DIR / 'test' / 'images').iterdir()
    if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
)
if MAX_IOU_IMAGES is not None:
    test_images = test_images[:MAX_IOU_IMAGES]

iou_summary_rows, iou_detail_rows = [], []
for model_name, checkpoint in CHECKPOINTS.items():
    print(f'\nIoU analysis: {model_name}')
    model, all_ious = YOLO(str(checkpoint)), []
    names = names_dict(model.names)
    for number, image_path in enumerate(test_images, 1):
        gt_boxes, gt_classes = ground_truth(image_path)
        result = model.predict(
            source=str(image_path), imgsz=IMGSZ, conf=DEPLOY_CONF,
            device=DEVICE, verbose=False,
        )[0]
        if result.boxes is None or len(result.boxes) == 0:
            continue
        values, classes = matched_ious(
            result.boxes.xyxy.detach().cpu().numpy(),
            result.boxes.cls.detach().cpu().numpy().astype(int),
            result.boxes.conf.detach().cpu().numpy(),
            gt_boxes, gt_classes, MATCH_IOU_THRESHOLD,
        )
        all_ious.extend(values)
        for iou, class_id in zip(values, classes):
            iou_detail_rows.append({
                'model': model_name, 'image': image_path.name, 'class_id': class_id,
                'class_name': names.get(class_id, str(class_id)), 'iou': iou,
            })
        if number % 100 == 0:
            print(f'{number}/{len(test_images)}')
    iou_summary_rows.append({
        'model': model_name, 'matched_true_positives': len(all_ious),
        'mean_matched_iou': float(np.mean(all_ious)) if all_ious else np.nan,
        'median_matched_iou': float(np.median(all_ious)) if all_ious else np.nan,
        'p10_matched_iou': float(np.percentile(all_ious, 10)) if all_ious else np.nan,
        'matching_threshold': MATCH_IOU_THRESHOLD,
        'confidence_threshold': DEPLOY_CONF,
    })

iou_summary_df = pd.DataFrame(iou_summary_rows)
iou_details_df = pd.DataFrame(iou_detail_rows)
display(iou_summary_df)
if not iou_details_df.empty:
    display(iou_details_df.groupby(['model', 'class_name'])['iou'].agg(['count', 'mean', 'median']).reset_index())

## 6. Batch-1 latency, FPS, and peak GPU-memory benchmark

In [ ]:
valid_name = 'valid' if (LOCAL_DATASET_DIR / 'valid').exists() else 'val'
benchmark_source_paths = sorted(
    path for path in (LOCAL_DATASET_DIR / valid_name / 'images').iterdir()
    if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
)
benchmark_paths = benchmark_source_paths[:min(BENCHMARK_IMAGES, len(benchmark_source_paths))]
benchmark_arrays = [np.asarray(Image.open(path).convert('RGB')) for path in benchmark_paths]
if not benchmark_arrays:
    raise RuntimeError('No test images found.')

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def benchmark_model(model: YOLO, images: list[np.ndarray]) -> dict:
    for index in range(WARMUP_RUNS):
        model.predict(source=images[index % len(images)], imgsz=IMGSZ, conf=DEPLOY_CONF, device=DEVICE, verbose=False)
    sync_cuda(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    wall, preprocessing, inference, postprocessing = [], [], [], []
    for image in images:
        sync_cuda(); start = time.perf_counter()
        result = model.predict(source=image, imgsz=IMGSZ, conf=DEPLOY_CONF, device=DEVICE, verbose=False)[0]
        sync_cuda(); wall.append((time.perf_counter() - start) * 1000)
        preprocessing.append(float(result.speed.get('preprocess', np.nan)))
        inference.append(float(result.speed.get('inference', np.nan)))
        postprocessing.append(float(result.speed.get('postprocess', np.nan)))
    median = float(np.median(wall))
    return {
        'benchmark_images': len(images),
        'median_wall_latency_ms': median,
        'mean_wall_latency_ms': float(np.mean(wall)),
        'p95_wall_latency_ms': float(np.percentile(wall, 95)),
        'fps_from_median_latency': 1000.0 / median,
        'mean_preprocess_ms': float(np.nanmean(preprocessing)),
        'mean_inference_ms': float(np.nanmean(inference)),
        'mean_postprocess_ms': float(np.nanmean(postprocessing)),
        'peak_vram_mb': float(torch.cuda.max_memory_allocated() / (1024 ** 2)),
    }

benchmark_rows = []
for model_name, checkpoint in CHECKPOINTS.items():
    print(f'\nBenchmarking {model_name}...')
    gc.collect(); torch.cuda.empty_cache()
    load_start = time.perf_counter(); model = YOLO(str(checkpoint))
    model_load_ms = (time.perf_counter() - load_start) * 1000
    sync_cuda(); cold_start = time.perf_counter()
    model.predict(source=benchmark_arrays[0], imgsz=IMGSZ, conf=DEPLOY_CONF, device=DEVICE, verbose=False)
    sync_cuda(); cold_ms = (time.perf_counter() - cold_start) * 1000
    row = benchmark_model(model, benchmark_arrays)
    row.update({'model': model_name, 'model_load_ms': model_load_ms, 'cold_first_inference_ms': cold_ms})
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows).sort_values('median_wall_latency_ms')
display(benchmark_df)

## 7. Combined comparison and trade-off chart

In [ ]:
comparison_df = test_metrics_df.merge(iou_summary_df, on='model', how='left').merge(benchmark_df, on='model', how='left')
comparison_df = comparison_df.sort_values(['map50_95', 'median_wall_latency_ms'], ascending=[False, True])
comparison_df.to_csv(COMPARISON_DIR / 'model_comparison.csv', index=False)
per_class_df.to_csv(COMPARISON_DIR / 'per_class_map50_95.csv', index=False)
iou_details_df.to_csv(COMPARISON_DIR / 'matched_iou_details.csv', index=False)
display(comparison_df)

In [ ]:
import matplotlib.pyplot as plt
ax = comparison_df.plot.scatter(x='median_wall_latency_ms', y='map50_95', figsize=(8, 6))
for _, row in comparison_df.iterrows():
    ax.annotate(row['model'], (row['median_wall_latency_ms'], row['map50_95']), xytext=(5, 5), textcoords='offset points')
ax.set_title('Accuracy versus batch-1 end-to-end latency')
ax.set_xlabel('Median wall-clock latency (ms)')
ax.set_ylabel('Test mAP@0.50:0.95')
plt.tight_layout(); plt.savefig(COMPARISON_DIR / 'accuracy_latency_tradeoff.png', dpi=180); plt.show()

## 8. Pareto-efficient candidates

In [ ]:
def pareto_mask(frame, accuracy='map50_95', latency='median_wall_latency_ms'):
    mask = []
    for _, row in frame.iterrows():
        dominated = ((frame[accuracy] >= row[accuracy]) & (frame[latency] <= row[latency]) &
                     ((frame[accuracy] > row[accuracy]) | (frame[latency] < row[latency]))).any()
        mask.append(not dominated)
    return mask

comparison_df['pareto_efficient'] = pareto_mask(comparison_df)
display(comparison_df[[
    'model', 'map50_95', 'recall', 'mean_matched_iou',
    'median_wall_latency_ms', 'fps_from_median_latency',
    'model_size_mb', 'peak_vram_mb', 'pareto_efficient',
]])

## 9. Export candidates to ONNX

In [ ]:
export_rows = []
if RUN_ONNX_EXPORT:
    for model_name, checkpoint in CHECKPOINTS.items():
        print(f'\nExporting {model_name}...')
        model = YOLO(str(checkpoint))
        exported = Path(model.export(format='onnx', imgsz=IMGSZ, batch=1, dynamic=False, simplify=True, opset=17))
        destination = EXPORTS_ROOT / f'{model_name}_road_damage.onnx'
        shutil.copy2(exported, destination)
        row = {'model': model_name, 'format': 'onnx', 'path': str(destination), 'size_mb': destination.stat().st_size / (1024 ** 2)}
        if VALIDATE_ONNX:
            metrics = YOLO(str(destination), task='detect').val(
                data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=1,
                device=DEVICE, workers=WORKERS, plots=False,
            )
            native = float(comparison_df.loc[comparison_df['model'] == model_name, 'map50_95'].iloc[0])
            row['onnx_map50_95'] = float(metrics.box.map)
            row['map50_95_drift'] = row['onnx_map50_95'] - native
        export_rows.append(row)
export_df = pd.DataFrame(export_rows)
if not export_df.empty:
    export_df.to_csv(COMPARISON_DIR / 'exports.csv', index=False)
    display(export_df)

## 10. Select and package the production model

In [ ]:
MIN_VALIDATION_RECALL = 0.50
MAX_P95_LATENCY_MS = 100.0
MAX_MODEL_SIZE_MB = 100.0

selection_df = validation_df.merge(
    benchmark_df,
    on='model',
    how='left',
)

eligible = selection_df[
    (selection_df['validation_recall'] >= MIN_VALIDATION_RECALL) &
    (selection_df['p95_wall_latency_ms'] <= MAX_P95_LATENCY_MS) &
    (selection_df['model_size_mb'] <= MAX_MODEL_SIZE_MB)
].copy()

if eligible.empty:
    print('No model meets the predeclared validation/deployment constraints.')
else:
    winner = eligible.sort_values(
        ['validation_map50_95', 'median_wall_latency_ms'],
        ascending=[False, True],
    ).iloc[0]
    winner_name = winner['model']
    winner_checkpoint = CHECKPOINTS[winner_name]

    production_pt = EXPORTS_ROOT / 'production_road_damage_model.pt'
    shutil.copy2(winner_checkpoint, production_pt)

    candidate_onnx = EXPORTS_ROOT / f'{winner_name}_road_damage.onnx'
    production_onnx = EXPORTS_ROOT / 'production_road_damage_model.onnx'
    if candidate_onnx.exists():
        shutil.copy2(candidate_onnx, production_onnx)

    test_report = comparison_df.loc[
        comparison_df['model'] == winner_name
    ].iloc[0].to_dict()

    decision = {
        'selected_model': winner_name,
        'selection_basis': 'validation metrics plus fixed-hardware benchmark; test metrics excluded',
        'checkpoint': str(winner_checkpoint),
        'production_pt': str(production_pt),
        'production_onnx': str(production_onnx) if production_onnx.exists() else None,
        'constraints': {
            'minimum_validation_recall': MIN_VALIDATION_RECALL,
            'maximum_p95_latency_ms': MAX_P95_LATENCY_MS,
            'maximum_model_size_mb': MAX_MODEL_SIZE_MB,
        },
        'selection_metrics': {
            key: (value.item() if isinstance(value, np.generic) else value)
            for key, value in winner.to_dict().items()
        },
        'held_out_test_report_after_selection': {
            key: (value.item() if isinstance(value, np.generic) else value)
            for key, value in test_report.items()
        },
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    (COMPARISON_DIR / 'production_model_decision.json').write_text(
        json.dumps(decision, indent=2), encoding='utf-8'
    )
    print(json.dumps(decision, indent=2))

## Completed

Results: `MyDrive/RoadDamageYOLO/reports/final_comparison`  
Exports: `MyDrive/RoadDamageYOLO/exports`

Continue with notebook 05 to generate the Streamlit app.

### Shared artifact contract

This notebook writes checkpoints and metrics to Google Drive. The later comparison
notebook discovers them through the fixed run-directory names. Do not manually rename:

- `YOLOv8s`, `YOLO11s`, or `YOLO26s`
- `baseline_seed42`
- `final_tuned_seed42`
- their `weights/best.pt` files